In [1]:
!pip install polars statsmodels prdc mauve-text sentence-transformers pybiber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 170.7 MB/s eta 0:00:00


In [2]:
import time
import random
import torch
import os
import sklearn
import re

import pandas as pd
import polars as pl
import numpy as np

import scipy
import statsmodels.api as sm
from statsmodels.formula.api import ols

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from pathlib import Path
import json

import torch
import gc

from itertools import combinations

In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("huggingFaceToken")
login(token=HF_TOKEN)

In [4]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/realDataAnalysis/ablation'

Mounted at /content/drive


In [5]:
import sys
sys.path.append(f'{DATA_DIR}')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder


In [6]:
# Remove transformers verbosity to clean up space.
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

# Silence HuggingFace
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Silence Python warnings.
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pybiber").setLevel(logging.ERROR)

In [7]:
# Make files for consistent saving.

BASE_OUTPUT = Path(f"{DATA_DIR}/outputCompcor")

DIRS = {
    "ksc_synth": BASE_OUTPUT / "ksc_synth",
    "ksc": BASE_OUTPUT / "ksc",
    # "size_imbalance": BASE_OUTPUT / "size_imbalance",
    "plots": BASE_OUTPUT / "plots",
}

# Make all directories.
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

PLOT_DIRS = {
    "ksc_synth": DIRS["plots"] / "ksc_synth",
    "ksc": DIRS["plots"] / "ksc",
    # "size_imbalance": DIRS["plots"] / "size_imbalance",
}

for d in PLOT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

In [8]:
# Set plotting visualization config options.
SMALL_SIZE = 10
matplotlib.rc('font', size=SMALL_SIZE)
matplotlib.rc('axes', titlesize=SMALL_SIZE)
sns.set_theme(style="whitegrid", font_scale=2)

In [9]:
# Add file name helper.
def make_filename(*parts, ext="csv"):
    clean = "_".join(str(p).replace("/", "-") for p in parts)
    return f"{clean}.{ext}"

# Add plot saving helper.
def save_plot(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [10]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    # os.environ["TOKENIZERS_PARALLELISM"] = "false" # done earlier
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [11]:
# ------------------ Metric Setup (experiment_config.py) ------------------
ksc_measures = ['Accuracy', 'Weighted Accuracy', 'Time', 'Monotonicity', 'Separability', 'Linearity']

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	c = corpus
	return c

In [12]:
# ------------------ Loading and summarizing data functions. (utils.py) ------------------

# Simple text cleaning function.
def preprocessing(texts):
    processed_texts = []
    for text in texts:
        text = str(text).strip()
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
        text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
        processed_texts.append(text)
    return processed_texts

# Helper function to load a labelled corpus.
def load_corpus(filename, sep=',', max_samples=np.inf):
    data = pd.read_csv(filename, sep=sep)
    data.drop(np.where(pd.isnull(data))[0], axis=0, inplace=True) # drop null data
    data = data.apply(lambda x: x.str.strip()) # strip extra whitespace
    data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True) # randomly shuffle dataset
    if not np.isinf(max_samples):
        data = data.head(max_samples) # get the number of samples required from the dataset
    sentences = data['text'].dropna().astype(str).tolist() # convert sentences to list and remove all NaN values
    return preprocessing(sentences)

def load_generated_corpus(filename, sep=',', max_samples=np.inf):
    data = pd.read_csv(filename, sep=sep)
    data.drop(np.where(pd.isnull(data))[0], axis=0, inplace=True) # drop null data
    data = data.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)) # strip extra whitespace
    data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True) # randomly shuffle dataset
    if not np.isinf(max_samples):
        data = data.head(max_samples) # get the number of samples required from the dataset
    sentences = data['report'].dropna().astype(str).tolist() # convert sentences to list and remove all NaN values
    return preprocessing(sentences)

def load_generated_and_real_data(max_samples=np.inf):
    atis = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/atis.csv', max_samples=max_samples)
    atis_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/atis.csv', max_samples=max_samples)

    banking77 = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/banking77.csv', max_samples=max_samples)
    banking77_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/banking77.csv', max_samples=max_samples)

    clinc150 = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/clinc150.csv', max_samples=max_samples)
    clinc150_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/clinc150.csv', max_samples=max_samples)

    clinicalDialogueSummarizations = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/clinicalDialogueSummarizations.csv', max_samples=max_samples)
    clinicalDialogueSummarizations_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/clinicalDialogueSummarizations.csv', max_samples=max_samples)

    dementiaAudio = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/dementiaAudio.csv', max_samples=max_samples)
    dementiaAudio_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/dementiaAudio.csv', max_samples=max_samples)

    huffPostNews = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/huffPostNews.csv', max_samples=max_samples)
    huffPostNews_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/huffPostNews.csv', max_samples=max_samples)

    medicalAbstracts = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/medicalAbstracts.csv', max_samples=max_samples)
    medicalAbstracts_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/medicalAbstracts.csv', max_samples=max_samples)

    simSUM = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/simSUM.csv', max_samples=max_samples)
    simSUM_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/simSUM.csv', max_samples=max_samples)

    syntheticCareHomeNurseNotes = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/syntheticCareHomeNurseNotes.csv', max_samples=max_samples)
    syntheticCareHomeNurseNotes_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/syntheticCareHomeNurseNotes.csv', max_samples=max_samples)

    yahoo = load_corpus(f'{DATA_DIR}/datasets/datasetsPrep/yahoo.csv', max_samples=max_samples)
    yahoo_gen = load_generated_corpus(f'{DATA_DIR}/datasets/generatedData/yahoo.csv', max_samples=max_samples)

    return (
        atis, atis_gen,
        banking77, banking77_gen,
        clinc150, clinc150_gen,
        clinicalDialogueSummarizations, clinicalDialogueSummarizations_gen,
        dementiaAudio, dementiaAudio_gen,
        huffPostNews, huffPostNews_gen,
        medicalAbstracts, medicalAbstracts_gen,
        simSUM, simSUM_gen,
        syntheticCareHomeNurseNotes, syntheticCareHomeNurseNotes_gen,
        yahoo, yahoo_gen
    )

# Helper function to summarize results.
def summarize_results(metrics_measures_df):
    mu = metrics_measures_df.groupby(['metric']).mean()
    mu = mu.round(decimals=3)
    std = metrics_measures_df.groupby(['metric']).std()
    return mu, std

In [13]:
# ------------------ Functions to compute metric characteristics. (metric_characteristics.py) ------------------

# Helper function for metric monotonicity.
def metric_monotonicity(ells, distances):
    return scipy.stats.spearmanr(ells, distances).correlation

# Helper function for metric separability.
def metric_separability(ells, distances):
    df = pd.DataFrame(data=list(zip(ells, distances)), columns=['ell', 'distance'])
    model = ols('distance ~ C(ell)', data=df).fit()
    aov_table = sm.stats.anova_lm(model, typ=2)
    return anova_table(aov_table).loc['C(ell)', 'omega_sq']

# Helper function for anova table.
def anova_table(aov):
    aov['mean_sq'] = aov[:]['sum_sq'] / aov[:]['df']
    aov['eta_sq'] = aov.iloc[:-1]['sum_sq'] / sum(aov['sum_sq'])
    aov['omega_sq'] = (aov.iloc[:-1]['sum_sq'] - (aov.iloc[:-1]['df'] * aov['mean_sq'].iloc[-1])) / (
                sum(aov['sum_sq']) + aov['mean_sq'].iloc[-1])
    cols = ['sum_sq', 'df', 'mean_sq', 'F', 'PR(>F)', 'eta_sq', 'omega_sq']
    aov = aov[cols]
    return aov

# Helper function for metric linearity.
def metric_linearity(ells, distances):
    return scipy.stats.linregress(ells, y=distances).rvalue

In [14]:
def known_similarity_corpora(sentences_set1, sentences_set2, n=50, k=5, unique_samples_corpora=True):
        """
        given 2 sets of sentences, creates k Known Similarity corpora of size n.
        :param sentences_set1: sentences of domain 1.
        :param sentences_set2: sentences of domain 1.
        :param n: length of the created corpora.
        :param k: the number of created corpora.
        :param unique_samples_corpora: maintain that each sample is used in a single corpus in the KSC
        :return: a set of sets containing the k created corpora.
        """
        if unique_samples_corpora and (len(sentences_set1)<k*n or len(sentences_set2)<k*n):
            raise Exception("To build KSC with n={} and k={} there should be at least {} items in the initial corpora which currenly contains: {},{}."
                            "(to ensure all items in all combination corpora should are different".format(n, k, k * n, len(sentences_set1),  len(sentences_set2)))

        sentences_set1_unused = np.array(sentences_set1)
        sentences_set2_unused = np.array(sentences_set2)

        ksc = []
        for i in np.linspace(0, 1, k, endpoint=True):
            p = int(i * n)
            current_set1_indx = random.sample(range(len(sentences_set1_unused)), p)
            current_set1 = sentences_set1_unused[current_set1_indx]
            if unique_samples_corpora:
                sentences_set1_unused = np.delete(sentences_set1_unused, current_set1_indx, axis=0)
            current_set2_indx = random.sample(range(len(sentences_set2_unused)), n-p)
            current_set2 = sentences_set2_unused[current_set2_indx]
            if unique_samples_corpora:
                sentences_set2_unused = np.delete(sentences_set2_unused, current_set2_indx, axis=0)
            ksc += [list(current_set1) + list(current_set2)]

        return ksc

In [15]:
def KSC_comparisons_rec(start, end):
    s = []
    if end - start == 1:
        return [], []
    son1 = (start, end - 1)
    son2 = (start + 1, end)
    s.append([son1, (start, end)])
    s.append([son2, (start, end)])
    right_judg, right_sons = KSC_comparisons_rec(son1[0], son1[1])
    left_judg, left_sons = KSC_comparisons_rec(son2[0], son2[1])
    s.extend(right_judg)
    s.extend(left_judg)
    all_sons = set(right_sons + left_sons)
    desc_judge = [[son, (start, end)] for son in all_sons]
    s.extend(desc_judge)
    return s, list(all_sons) + [son1] + [son2]

In [16]:
def KSC_comparisons(k):
    s, _ = KSC_comparisons_rec(0, k-1)
    return s

In [17]:
ZERO_SHOT_MODELS = [
"cross-encoder/nli-deberta-v3-small", # low capacity
"typeform/distilbert-base-uncased-mnli", # medium capacity
"valhalla/distilbart-mnli-12-3", # higher capacity
]

# prompt templates
TEMPLATES = {
	"prompt1": "This example is {}.",
	"prompt2": "The writing style of this text is {}.",
	"prompt3": "This text is {}.",
	"prompt4": "This text is written in a {} style.",
	"prompt5": "This text shows {} characteristics."
}


# Make combinations (every possible non-empty model subset)
def all_nonempty_subsets(items):
	return [
		list(combo)
		for r in range(1, len(items) + 1) for combo in combinations(items, r)
	]

model_combinations = all_nonempty_subsets(ZERO_SHOT_MODELS)
prompt_combinations = all_nonempty_subsets(TEMPLATES.keys())

In [18]:
def runKSC(d1, d2,  output_dir, n, k, repetitions, output_name):
  ksc_results = []
  distance_results = []

  output_dir.mkdir(parents=True, exist_ok=True)

  with torch.no_grad():
      c1 = get_metric_dependant_data(corpus_metrics.zero_wasserstein_distance, d1)
      c2 = get_metric_dependant_data(corpus_metrics.zero_wasserstein_distance, d2)
  if torch.is_tensor(c1):
      c1 = c1.detach().cpu()
  if torch.is_tensor(c2):
      c2 = c2.detach().cpu()
  for repit in range(repetitions):
    with torch.no_grad():
        ksc = known_similarity_corpora(c1, c2, n=n, k=k, unique_samples_corpora=True)
    with torch.no_grad():
      ksc_doc = []
      for corpus in ksc:
        ksc_doc.append(corpus)

      all_distance_stats = {}

      for i in range(len(ksc)):
          for j in range(i + 1, len(ksc)):
              all_metrics = corpus_metrics.zero_wasserstein_distance(ksc_doc[i], ksc_doc[j])
              all_distance_stats[f"{i}_{j}"] = all_metrics

      all_comparisons = KSC_comparisons(len(ksc))
      for models in model_combinations:
        for prompts in prompt_combinations:
          distances_metric = []
          distance_stats = []
          d_ksc = np.ones([len(ksc), len(ksc)]) * np.nan
          all_times = []
          for i in range(len(ksc)):
            for j in range(i + 1, len(ksc)):
              temp_df = all_distance_stats[f"{i}_{j}"]
              metric_df = temp_df[temp_df['models'].apply(lambda x: x == models) & temp_df['prompts'].apply(lambda x: x == prompts)]
              d_ksc[i, j] = float(metric_df['wasserstein'])
              distance_stats.append((i, j, d_ksc[i, j]))
              all_times.append(metric_df['time'])
          metric_name = f"{'_'.join(models)}__{'_'.join(prompts)}"
          comparisons_results = []
          print("Num judgments: {}".format(len(all_comparisons)))
          for comparison in all_comparisons:
              i = comparison[0][0]
              j = comparison[0][1]
              m = comparison[1][0]
              n = comparison[1][1]
              d_ij = d_ksc[i, j]
              d_mn = d_ksc[m, n]
              comparisons_results.append([(i, j), (m, n), 1 if d_ij < d_mn else 0])

          comparison_accuracy = np.array([x[2] for x in comparisons_results])

          d0 = [j - i for ((i, j), _, _) in comparisons_results]
          d1 = [n - m for (_, (m, n), _) in comparisons_results]
          weights = 1 / (np.array(d1) - np.array(d0))

          weighted_comparison_accuracy = np.sum(comparison_accuracy * (weights)) / np.sum(weights)
          print(
              "{}: KSC_Score: {},Weighted KSC:{}".format(metric_name, np.mean(comparison_accuracy),
                                                          weighted_comparison_accuracy))
          ksc_time = np.mean(all_times)

          distances_metric.append(np.vstack([[metric_name, repit, a, b, b - a, y] for (a, b, y) in distance_stats]))

          distances_metric = np.vstack(distances_metric)

          # normalize the score for a specific metric.
          distances_metric = np.append(distances_metric, sklearn.preprocessing.StandardScaler().fit_transform(distances_metric[:, 5].reshape(-1, 1)), axis=1)
          distance_results.extend(distances_metric)

          ells = distances_metric[:, 4].astype('float')
          ds_normalized = distances_metric[:, 6].astype('float')

          monotonicity = metric_monotonicity(ells, ds_normalized)
          separability = metric_separability(ells, ds_normalized)
          linearity = metric_linearity(ells, ds_normalized)
          ksc_results.append(
              [metric_name, np.mean(comparison_accuracy), weighted_comparison_accuracy, ksc_time, monotonicity, separability, linearity])


  gc.collect()
  torch.cuda.empty_cache()
  del c1, c2, ksc

  metrics_measures_df = pd.DataFrame(data=ksc_results, columns=['metric'] + ksc_measures)
  metrics_measures_df['Time'] = (1 / metrics_measures_df['Time'])/100

  all_distance_samples_df = pd.DataFrame(data=distance_results,
                                  columns=['metric', 'repetition', 'i', 'j', 'l', 'distance', 'distance_score'])
  all_distance_samples_df["l"] = pd.to_numeric(all_distance_samples_df["l"])
  all_distance_samples_df["distance"] = pd.to_numeric(all_distance_samples_df["distance"])
  all_distance_samples_df["distance_score"] = pd.to_numeric(all_distance_samples_df["distance_score"])
  metrics_measures_df.to_csv(
  output_dir / make_filename(f"{output_name}_ksc_metrics_measures"), index=False
  )

  all_distance_samples_df.to_csv(
      output_dir / make_filename(f"{output_name}_ksc_distance_samples"), index=False
  )
  return metrics_measures_df, all_distance_samples_df


def plotKSC(all_distance_samples_df, save_path = None, output_name='test'):
    metrics_names = np.unique(all_distance_samples_df['metric'])
    fig, axlist = plt.subplots(1, len(metrics_names), figsize=(35, 5))
    if len(metrics_names) == 1:
        axlist = [axlist]
    for i, metric in enumerate(metrics_names):
        metric_df = all_distance_samples_df[all_distance_samples_df['metric'] == metric]
        sns.scatterplot(x='l', y='distance', data=metric_df, ax=axlist[i], color='orange')
        sns.regplot(x='l', y='distance', data=metric_df, ax=axlist[i],
                    scatter=False, truncate=False)
        axlist[i].set_title('{}'.format(metric))
        axlist[i].set_xlabel('')
        axlist[i].set_ylabel('')

    plt.subplots_adjust(left=0.05,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

    if save_path:
        save_plot(fig, save_path / f"{output_name}_ksc_distance_plot.png")
    else:
        plt.show()


def plot_measures_results(metrics_measures_df, save_path = None, output_name='test'):
    fig, ax = plt.subplots(1, 6, figsize=(35, 5))
    if isinstance(ax, np.ndarray):
        ax = ax.flatten()
    else:
        ax = [ax]
    for i, measure in enumerate(ksc_measures):
        sns.boxplot(ax=ax[i], x='metric', y=measure, data=metrics_measures_df)
        ax[i].set_xlabel('')
        ax[i].tick_params(axis='x', labelsize=5)

    plt.subplots_adjust(left=0.1,
                        bottom=0.1,
                        right=0.99,
                        top=0.9,
                        wspace=0.3,
                        hspace=0.4)

    if save_path:
        save_plot(fig, save_path / f"{output_name}_ksc_measures_boxplot.png")
    else:
        plt.show()

In [19]:
n_samples = 100
# n_samples = 40 # as the smallest real dataset tested is 549 data points long, so 40 x 12 will be 480
L = [n_samples, 7]
H = [n_samples, 12]
rep = 5

max_samples = H[0] * H[1] * rep

# Make data for KSC experiment.
(atis, atis_gen,
banking77, banking77_gen,
clinc150, clinc150_gen,
clinicalDialogueSummarizations, clinicalDialogueSummarizations_gen,
dementiaAudio, dementiaAudio_gen,
huffPostNews, huffPostNews_gen,
medicalAbstracts, medicalAbstracts_gen,
simSUM, simSUM_gen,
syntheticCareHomeNurseNotes, syntheticCareHomeNurseNotes_gen,
yahoo, yahoo_gen
) = load_generated_and_real_data(max_samples)

# Make dictionaries.
real_datasets = [
    ('atis', atis),
    ('banking77', banking77),
    ('clinc150', clinc150),
    ('clinicalDialogueSummarizations', clinicalDialogueSummarizations),
    # ('dementiaAudio', dementiaAudio),
    ('huffPostNews', huffPostNews),
    ('medicalAbstracts', medicalAbstracts),
    ('simSUM', simSUM),
    ('syntheticCareHomeNurseNotes', syntheticCareHomeNurseNotes),
    ('yahoo', yahoo),
]

gen_datasets = [
    ('atis_gen', atis_gen),
    ('banking77_gen', banking77_gen),
    ('clinc150_gen', clinc150_gen),
    ('clinicalDialogueSummarizations_gen', clinicalDialogueSummarizations_gen),
    # ('dementiaAudio_gen', dementiaAudio_gen),
    ('huffPostNews_gen', huffPostNews_gen),
    ('medicalAbstracts_gen', medicalAbstracts_gen),
    ('simSUM_gen', simSUM_gen),
    ('syntheticCareHomeNurseNotes_gen', syntheticCareHomeNurseNotes_gen),
    ('yahoo_gen', yahoo_gen),
]


real_pairs = list(combinations(real_datasets, 2))
real_gen_pairs = list(zip(real_datasets, gen_datasets))

for R in [L,H]:
  for (name1, d1), (name2, d2) in real_pairs:
    results_file_name = DIRS["ksc"] / f"{name1}_{name2}_{R[0]}_{R[1]}"
    output_name = f"{name1}_{name2}_{R[0]}_{R[1]}"

    check_file = PLOT_DIRS["ksc"] / f"{output_name}_ksc_measures_boxplot.png"
    if check_file.exists():
      print(f"Skipping {output_name} (already exists)")
      continue

    metrics_measures_df, all_distance_samples_df = runKSC(d1, d2,  output_dir=results_file_name, n=R[0], k=R[1], repetitions=rep, output_name = output_name)
    plotKSC(all_distance_samples_df, save_path=PLOT_DIRS["ksc"], output_name = output_name)
    plot_measures_results(metrics_measures_df, save_path=PLOT_DIRS["ksc"], output_name = output_name)
    mu12, std12 = summarize_results(metrics_measures_df)

    del metrics_measures_df, all_distance_samples_df
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    gc.collect()

for R in [L,H]:
  for (name1, d1), (name2, d2) in real_gen_pairs:
    results_file_name = DIRS["ksc_synth"] / f"{name1}_{name2}_{R[0]}_{R[1]}"
    output_name = f"{name1}_{name2}_{R[0]}_{R[1]}"

    check_file = PLOT_DIRS["ksc_synth"] / f"{output_name}_ksc_measures_boxplot.png"
    if check_file.exists():
      print(f"Skipping {output_name} (already exists)")
      continue

    metrics_measures_df, all_distance_samples_df = runKSC(d1, d2,  output_dir=results_file_name, n=R[0], k=R[1], repetitions=rep, output_name = output_name)
    plotKSC(all_distance_samples_df, save_path=PLOT_DIRS["ksc_synth"], output_name = output_name)
    plot_measures_results(metrics_measures_df, save_path=PLOT_DIRS["ksc_synth"], output_name = output_name)
    mu12, std12 = summarize_results(metrics_measures_df)

    del metrics_measures_df, all_distance_samples_df
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    gc.collect()

Skipping atis_banking77_100_7 (already exists)
Skipping atis_clinc150_100_7 (already exists)
Skipping atis_clinicalDialogueSummarizations_100_7 (already exists)
Skipping atis_huffPostNews_100_7 (already exists)
Skipping atis_medicalAbstracts_100_7 (already exists)
Skipping atis_simSUM_100_7 (already exists)
Skipping atis_syntheticCareHomeNurseNotes_100_7 (already exists)
Skipping atis_yahoo_100_7 (already exists)
Skipping banking77_clinc150_100_7 (already exists)
Skipping banking77_clinicalDialogueSummarizations_100_7 (already exists)
Skipping banking77_huffPostNews_100_7 (already exists)
Skipping banking77_medicalAbstracts_100_7 (already exists)
Skipping banking77_simSUM_100_7 (already exists)
Skipping banking77_syntheticCareHomeNurseNotes_100_7 (already exists)
Skipping banking77_yahoo_100_7 (already exists)
Skipping clinc150_clinicalDialogueSummarizations_100_7 (already exists)
Skipping clinc150_huffPostNews_100_7 (already exists)
Skipping clinc150_medicalAbstracts_100_7 (already ex